In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from pathlib import Path

DATA = Path('/content/drive/MyDrive/Brain_alignment/lpp_data')

# Load Day 1 output
alignment = pd.read_csv(DATA / 'word_TR_alignment_run15.csv')
print(f"Loaded: {len(alignment)} words, {alignment['TR_index'].nunique()} TRs")

Mounted at /content/drive
Loaded: 1583 words, 286 TRs


In [3]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import GPT2Tokenizer, GPT2Model
from tqdm import tqdm

DATA  = Path('/content/drive/MyDrive/Brain_alignment/lpp_data')
ANNOT = DATA / 'annotation/EN'

alignment = pd.read_csv(DATA / 'word_TR_alignment_run15.csv')
print(f"Words: {len(alignment)}, TRs: {alignment['TR_index'].nunique()}")
print(alignment[['word','onset','pos','TR_index']].head())

Words: 1583, TRs: 286
      word  onset    pos  TR_index
0     good  0.015  PROPN         0
1  morning  0.130   NOUN         0
2     said  0.914   VERB         0
3      the  1.089    DET         0
4   little  1.169    ADJ         0


In [2]:
#make sure the gpu is connected before running the checkpoitn

import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

True Tesla T4


In [4]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2-medium')
model     = GPT2Model.from_pretrained('gpt2-medium')
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"Model on: {device}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.52GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Model on: cuda


In [5]:
def get_embeddings_context_sweep(words, context_sizes, layer=9):
    """
    For each word, extract GPT-2 hidden state at `layer`
    using different context window sizes.

    Returns dict: {context_size: np.array of shape (n_words, hidden_dim)}
    """
    results = {c: [] for c in context_sizes}

    for i, word in enumerate(tqdm(words, desc="Extracting")):
        for ctx in context_sizes:
            # Build context window: up to ctx words before current word
            start    = max(0, i - ctx) if ctx != -1 else 0  # -1 = full context
            ctx_words = words[start:i+1]
            text      = ' '.join(ctx_words)

            inputs = tokenizer(text, return_tensors='pt').to(device)

            # Guard: GPT-2 max 1024 tokens
            if inputs['input_ids'].shape[1] > 1024:
                inputs = {k: v[:, -1024:] for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs, output_hidden_states=True)

            # Take hidden state of the LAST token (current word) at target layer
            hidden = outputs.hidden_states[layer]  # (1, seq_len, 1024)
            word_emb = hidden[0, -1, :].cpu().numpy()  # (1024,)
            results[ctx].append(word_emb)

    return {c: np.array(v) for c, v in results.items()}

In [6]:
words = alignment['word'].tolist()

# Context sizes: 1, 5, 20, 50, 100, full (-1)
CONTEXT_SIZES = [1, 5, 20, 50, 100, -1]

print("Extracting embeddings... (~10-15 min on GPU, longer on CPU)")
embeddings = get_embeddings_context_sweep(words, CONTEXT_SIZES, layer=9)

for ctx, emb in embeddings.items():
    label = 'full' if ctx == -1 else str(ctx)
    print(f"  context={label}: shape {emb.shape}")

Extracting embeddings... (~10-15 min on GPU, longer on CPU)


Extracting: 100%|██████████| 1583/1583 [07:31<00:00,  3.51it/s]

  context=1: shape (1583, 1024)
  context=5: shape (1583, 1024)
  context=20: shape (1583, 1024)
  context=50: shape (1583, 1024)
  context=100: shape (1583, 1024)
  context=full: shape (1583, 1024)


In [7]:
N_TRS  = 286
HIDDEN = 1024

tr_embeddings = {}

for ctx, emb in embeddings.items():
    label = 'full' if ctx == -1 else str(ctx)
    tr_emb = np.zeros((N_TRS, HIDDEN))
    counts = np.zeros(N_TRS)

    for i, tr_idx in enumerate(alignment['TR_index'].values):
        tr_emb[tr_idx]  += emb[i]
        counts[tr_idx]  += 1

    # Average (TRs with no words keep zero vector)
    mask = counts > 0
    tr_emb[mask] /= counts[mask, None]
    tr_embeddings[label] = tr_emb
    print(f"  context={label}: TR matrix shape {tr_emb.shape}, "
          f"TRs with words: {mask.sum()}")

# Save
out_path = DATA / 'gpt2_tr_embeddings_run15.npz'
np.savez(out_path, **tr_embeddings)
print(f"\nSaved: {out_path}")

  context=1: TR matrix shape (286, 1024), TRs with words: 286
  context=5: TR matrix shape (286, 1024), TRs with words: 286
  context=20: TR matrix shape (286, 1024), TRs with words: 286
  context=50: TR matrix shape (286, 1024), TRs with words: 286
  context=100: TR matrix shape (286, 1024), TRs with words: 286
  context=full: TR matrix shape (286, 1024), TRs with words: 286

Saved: /content/drive/MyDrive/Brain_alignment/lpp_data/gpt2_tr_embeddings_run15.npz


In [8]:
loaded = np.load(DATA / 'gpt2_tr_embeddings_run15.npz')
print("Keys:", list(loaded.keys()))
for k in loaded.keys():
    arr = loaded[k]
    print(f"  context={k}: shape={arr.shape}, mean={arr.mean():.4f}, "
          f"nonzero rows: {(arr.sum(axis=1) != 0).sum()}")

Keys: ['1', '5', '20', '50', '100', 'full']
  context=1: shape=(286, 1024), mean=-0.2199, nonzero rows: 286
  context=5: shape=(286, 1024), mean=-0.2000, nonzero rows: 286
  context=20: shape=(286, 1024), mean=-0.1580, nonzero rows: 286
  context=50: shape=(286, 1024), mean=-0.1273, nonzero rows: 286
  context=100: shape=(286, 1024), mean=-0.1044, nonzero rows: 286
  context=full: shape=(286, 1024), mean=0.0353, nonzero rows: 286
